In [ ]:
# ==========================================
# CELL 1: ALL IMPORTS
# ==========================================
# Data Manipulation & Visualization
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import ScalarFormatter

# Preprocessing & Core ML Tools
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc, silhouette_score

# Supervised Models
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.naive_bayes import GaussianNB

# Unsupervised Models
from sklearn.cluster import KMeans, DBSCAN

# Environment settings
import warnings
warnings.filterwarnings('ignore')

print("All libraries imported successfully!")

In [ ]:
# ==========================================
# CELL 2: DATA SPLITTING & SCALING
# ==========================================
file_path = "Input_Data/01_cleaned_data/uwb_dataset_classification.parquet"

df = pd.read_parquet(file_path) 
X = df.drop('NLOS', axis=1)
y = df['NLOS']

# 1. Split: 80% Train, 20% Test (Balanced)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

# 2. Scale: Fit ONLY on training data, transform both
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(f"Data Pipeline Ready!")
print(f"Training Features: {X_train.shape} | Testing Features: {X_test.shape}")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc

def plot_independent_dashboard(model_name, grid, param_name):
    # Extract results
    best_model = grid.best_estimator_
    best_param_val = grid.best_params_[param_name]
    cv_res = pd.DataFrame(grid.cv_results_)
    
    # Get predictions only for the best model for Matrix and ROC
    y_pred = best_model.predict(X_test)
    y_prob = best_model.predict_proba(X_test)[:, 1]
    
    # --- UNIVERSAL FIX 1: Handle "None" / NaN values safely ---
    raw_params = cv_res[f'param_{param_name}'].values
    valid_nums = [x for x in raw_params if pd.notna(x)] # Filter out None/NaN
    
    # Create a safe placeholder for 'None' so math doesn't crash
    if valid_nums:
        placeholder = max(valid_nums) + 5 if max(valid_nums) >= 1 else max(valid_nums) * 1.5
    else:
        placeholder = 1
        
    numeric_params = [placeholder if pd.isna(x) else float(x) for x in raw_params]
    labels = ["None" if pd.isna(x) else str(x) for x in raw_params]
    
    # Create Layout
    fig, axes = plt.subplots(1, 3, figsize=(22, 6))
    best_label = "None" if pd.isna(best_param_val) else best_param_val
    fig.suptitle(f"Model Analysis: {model_name} | Best {param_name}: {best_label}", 
                 fontsize=18, fontweight='bold', y=1.05)
    
    # Graph 1: Accuracy Flow (All Hyperparameters)
    axes[0].plot(numeric_params, cv_res['mean_train_score'], marker='s', label='Train Accuracy', color='skyblue', lw=2)
    axes[0].plot(numeric_params, cv_res['mean_test_score'], marker='o', label='Test Accuracy (CV)', color='salmon', lw=2)
    
    # --- UNIVERSAL FIX 2: Auto-Detect Scale ---
    # Log scale is only used for parameters that span magnitudes (C, alpha, gamma, learning_rate)
    # Linear scale is used for integers (n_neighbors, max_depth, n_estimators)
    log_params = ['C', 'alpha', 'gamma', 'learning_rate']
    
    if param_name in log_params:
        axes[0].set_xscale('log') 
        axes[0].xaxis.set_major_formatter(ScalarFormatter())
    else:
        axes[0].set_xscale('linear')
        
    # Apply the safe ticks and labels
    axes[0].set_xticks(numeric_params)
    axes[0].set_xticklabels(labels, rotation=45)
    
    axes[0].set_title(f"Accuracy Curve (All {param_name})")
    axes[0].set_xlabel(f"Hyperparameter: {param_name}")
    axes[0].set_ylabel("Accuracy")
    axes[0].legend()
    
    # Graph 2: Confusion Matrix (Best Hyperparameter Only)
    cm = confusion_matrix(y_test, y_pred)
    ConfusionMatrixDisplay(cm).plot(ax=axes[1], cmap='Blues', colorbar=False)
    axes[1].set_title(f"Confusion Matrix (Best {param_name}={best_label})")
    
    # Graph 3: ROC Curve (Best Hyperparameter Only)
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    axes[2].plot(fpr, tpr, color='darkorange', label=f'AUC = {auc(fpr, tpr):.3f}')
    axes[2].plot([0, 1], [0, 1], color='navy', linestyle='--')
    axes[2].set_title(f"ROC Curve (Best {param_name}={best_label})")
    axes[2].legend(loc="lower right")
    
    plt.tight_layout()
    plt.show()

### GridSearchCV Selects the Best Hyperparameter

1. **The Evaluation:** For each parameter value (e.g., C), GridSearchCV runs cross-validation and records validation scores.
2. **The Averaging:** It computes the mean score across all folds to check consistent performance.
3. **The Selection:** The parameter with the highest average score (mean_test_score) is chosen.
4. **The Final Fit:** Using this best parameter, it retrains one final model on the full training set.

### Logistic Regression (Linear Models)

#### Step 1: The Linear Math (The "Regression" part)
The model calculates a weighted sum of input features:

**Equation:** $z = (Weight_1 \times Feature_1) + (Weight_2 \times Feature_2) + ... + Bias$

#### Step 2: The Sigmoid Function (The "Logistic" part)
The result \( z \) goes through the sigmoid function to map it between 0 and 1 — representing a **PROBABILITY**.

**Equation:** $Probability (p) = \frac{1}{1 + e^{-z}}$

#### Step 3: The Decision Boundary (Threshold)
The model predicts:
* If Probability $\ge 0.5 \rightarrow$ Predict Class 1 (Positive class)
* If Probability $< 0.5 \rightarrow$ Predict Class 0 (Negative class)

#### Step 4: How it Learns (Log Loss)
During training, it adjusts weights to minimize log loss, improving probability accuracy over iterations.

In [ ]:
from sklearn.linear_model import LogisticRegression

param_grid = {'C': [0.0005,0.0007,0.0009, 0.001, 0.002, 0.004, 0.006, 0.008, 0.01, 0.02, 0.03, 0.04, 0.05, 0.1]}

# 1. Define grid_lr
grid_lr = GridSearchCV(
    LogisticRegression(max_iter=1000, random_state=42), 
    param_grid, 
    cv=3, 
    return_train_score=True, 
    n_jobs=-1,
)

# 2. Fit using grid_lr
grid_lr.fit(X_train, y_train)

# 3. Plot using grid_lr
plot_independent_dashboard("Logistic Regression", grid_lr, 'C')

print("--- Average CV Scores ---")
results_lr = grid_lr.cv_results_

# Loop through the parameters and their corresponding average scores
for i in range(len(results_lr['params'])):
    c_val = results_lr['params'][i]['C']
    avg_score = results_lr['mean_test_score'][i] * 100 # Convert to percentage
    
    print(f"C = {c_val:<6} | Average Accuracy: {avg_score:.2f}%")

### Gaussian Naive Bayes (Probabilistic Models)

*A probabilistic classifier based on Bayes' Theorem. "Naive" assumes all features are independent; "Gaussian" assumes continuous features follow a normal distribution.*

#### 1. Prior Probability
Calculates the baseline probability of each class from training data:

$$P(y) = \frac{\text{count of class } y}{\text{total samples}}$$

(e.g., 70 out of 100 samples are Class A → P(A) = 0.70)

#### 2. Training
For each feature per class, it computes the **mean** and **variance**, fitting a Gaussian bell curve to each feature-class combination.

#### 3. Likelihood
For a new data point, it computes how probable each feature value is using the Gaussian probability formula:

$$P(x_i | y) = \frac{1}{\sigma\sqrt{2\pi}} \, e^{-\frac{(x_i - \mu)^2}{2\sigma^2}}$$

where $\mu$ is the mean and $\sigma$ is the standard deviation for that feature-class pair.

#### 4. Naive Multiplication (Posterior)
Multiplies the prior by all feature likelihoods independently per class using Bayes' Theorem:

$$P(y | X) \propto P(y) \cdot \prod_{i=1}^{n} P(x_i | y)$$

#### 5. Prediction
The class with the highest posterior probability is the final prediction:

$$\hat{y} = \arg\max_{y} \; P(y) \cdot \prod_{i=1}^{n} P(x_i | y)$$

In [ ]:
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import GridSearchCV

# Logarithmic range from very tiny (default is 1e-9) up to 1.0
param_grid = {'var_smoothing': [1e-9, 1e-8, 1e-7, 1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1]}

grid_nb = GridSearchCV(
    GaussianNB(), 
    param_grid, 
    cv=3, 
    return_train_score=True, 
    n_jobs=-1
)

print("Training Naive Bayes...")
grid_nb.fit(X_train, y_train)

# Output results using your universal function
plot_independent_dashboard("Gaussian Naive Bayes", grid_nb, 'var_smoothing')


print("--- Average CV Scores ---")
results_nb = grid_nb.cv_results_

# Loop through the parameters and their corresponding average scores
for i in range(len(results_nb['params'])):
    c_val = results_nb['params'][i]['var_smoothing']
    avg_score = results_nb['mean_test_score'][i] * 100 # Convert to percentage
    
    print(f"C = {c_val:<6} | Average Accuracy: {avg_score:.2f}%")

### Decision Tree Classifier (Tree-Based Models)

*A flowchart-like algorithm that classifies data by asking a series of Yes/No questions. Highly interpretable as it mimics human decision-making.*

#### 1. Finding the Best Split (Root)
Evaluates every feature to find the question (e.g., "Is Age ≤ 30?") that best separates the data into two distinct groups.

#### 2. Measuring Purity (Gini Impurity)
Uses **Gini Impurity** to measure how mixed each group is after a split:

$$Gini = 1 - \sum_{k=1}^{K} p_k^2$$

where $p_k$ is the proportion of class $k$ in the node. A score of 0.0 means perfect purity. The split with the **lowest** Gini score is chosen.

#### 3. Recursive Splitting
Repeats the same splitting process on each resulting group, growing the tree deeper with each iteration.

#### 4. Stopping Conditions (Leaves)
The tree stops growing when:
- The node is 100% pure (one class only)
- A limit is reached (e.g., `max_depth=5`)

These final nodes are called **Leaves**.

#### 5. Prediction
A new data point starts at the root, follows the Yes/No branches, and lands in a Leaf. The **majority class** of that Leaf is the final prediction:

$$\hat{y} = \arg\max_{k} \; p_k$$

In [ ]:
from sklearn.tree import DecisionTreeClassifier
param_grid = {'max_depth': [5,10, 15,20, 30, None]}
grid_dt = GridSearchCV(DecisionTreeClassifier(random_state=42), param_grid, cv=3, return_train_score=True, n_jobs=-1)
grid_dt.fit(X_train, y_train)
plot_independent_dashboard("Decision Tree", grid_dt, 'max_depth')

print("--- Average CV Scores ---")
results_dt = grid_dt.cv_results_

# Loop through the parameters and their corresponding average scores
for i in range(len(results_dt['params'])):
    c_val = results_dt['params'][i]['max_depth']
    avg_score = results_dt['mean_test_score'][i] * 100 # Convert to percentage
    
    print(f"C = {str(c_val):<6} | Average Accuracy: {avg_score:.2f}%")

### Random Forest Classifier (Tree-Based Models)

*An ensemble method that builds many Decision Trees and combines their predictions via majority vote — reducing the overfitting problem of a single Decision Tree.*

#### 1. Bootstrapping (Random Data)
Creates multiple mini-datasets by randomly sampling rows from the training data **with replacement**. Each tree trains on its own unique subset — unlike a single Decision Tree which uses the full dataset.

#### 2. Random Feature Selection (Random Splits)
At every split, each tree only considers a **random subset of features** (instead of all features like a standard Decision Tree). This forces diversity across trees and prevents one dominant feature from controlling all splits.

#### 3. Growing the Forest
Combines Steps 1 and 2 to grow many independent Decision Trees. The number of trees is set by `n_estimators` (default = 100). Each tree is intentionally different due to the randomness introduced.

#### 4. Prediction (Majority Vote)
A new data point is passed through **every** tree. Each tree independently predicts a class (just like a single Decision Tree would), then the forest takes a majority vote:

$$\hat{y} = \arg\max_{k} \sum_{t=1}^{T} \mathbf{1}[\hat{y}_t = k]$$

where $T$ is the total number of trees and $\hat{y}_t$ is the prediction of tree $t$. The class with the most votes wins.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
param_grid = {'n_estimators': [5,10,15,20,30,40, 50, 100]}
grid_rf = GridSearchCV(RandomForestClassifier(random_state=42, n_jobs=-1), param_grid, cv=3, return_train_score=True, n_jobs=-1)
grid_rf.fit(X_train, y_train)
plot_independent_dashboard("Random Forest", grid_rf, 'n_estimators')

print("--- Average CV Scores ---")
results_rf = grid_rf.cv_results_

# Loop through the parameters and their corresponding average scores
for i in range(len(results_rf['params'])):
    c_val = results_rf['params'][i]['n_estimators']
    avg_score = results_rf['mean_test_score'][i] * 100 # Convert to percentage
    
    print(f"C = {c_val:<6} | Average Accuracy: {avg_score:.2f}%")

### Gradient Boosting Classifier (Tree-Based Models)

*Unlike Random Forest (which builds trees independently), Gradient Boosting builds trees **sequentially** — each new Decision Tree specifically corrects the errors of all previous trees combined.*

#### 1. Initial Guess
Starts with a single flat prediction for all data points (e.g., overall majority class probability) — no Decision Tree is used yet.

#### 2. Calculate Residuals (Errors)
Measures how wrong the current prediction is for every data point:

$$r = y - \hat{p}$$

where $y$ is the true label and $\hat{p}$ is the current predicted probability.

#### 3. Fit a Decision Tree to the Errors
Builds a **shallow** Decision Tree — but unlike a standard Decision Tree predicting the true label, this tree is trained to predict the **residuals** from Step 2. It learns where the model is going wrong.

#### 4. Update with Learning Rate
Adds the new tree's predictions to the running total, scaled by a learning rate $\eta$ (e.g., 0.1) to avoid over-correcting:

$$\hat{p}_{\text{new}} = \hat{p}_{\text{old}} + \eta \cdot \hat{r}_t$$

#### 5. Repeat
Recalculates residuals, fits another shallow Decision Tree to those new errors, and repeats for $T$ iterations. The final prediction is:

$$\hat{y} = \hat{p}_0 + \sum_{t=1}^{T} \eta \cdot h_t(x)$$

where $h_t(x)$ is the prediction of the $t$-th Decision Tree.

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
param_grid = {'n_estimators': [10, 50, 100]}
grid_gb = GridSearchCV(GradientBoostingClassifier(random_state=42), param_grid, cv=3, return_train_score=True, n_jobs=-1)
grid_gb.fit(X_train, y_train)
plot_independent_dashboard("Gradient Boosting", grid_gb, 'n_estimators')

print("--- Average CV Scores ---")
results_gb = grid_gb.cv_results_

# Loop through the parameters and their corresponding average scores
for i in range(len(results_gb['params'])):
    c_val = results_gb['params'][i]['n_estimators']
    avg_score = results_gb['mean_test_score'][i] * 100 # Convert to percentage
    
    print(f"C = {c_val:<6} | Average Accuracy: {avg_score:.2f}%")

### XGBoost (Tree-Based Models)

*An optimized version of Gradient Boosting — faster, more accurate, and better at avoiding overfitting.*

#### 1. Sequential Learning
Builds Decision Trees one after another, where each new tree corrects the **residuals** (errors) of all previous trees — same core idea as Gradient Boosting.

#### 2. Regularization
Adds **L1 and L2 regularization** to the loss function to penalize overly complex trees and reduce overfitting:

$$L = \sum_{i} l(y_i, \hat{y}_i) + \sum_{t} \left( \gamma T + \frac{1}{2} \lambda \|w\|^2 \right)$$

where $T$ = number of leaves, $w$ = leaf weights, $\gamma$ and $\lambda$ = regularization terms.

#### 3. Optimized Split Finding
Uses a **Weighted Quantile Sketch** and parallel processing to evaluate splits efficiently — far faster than standard Gradient Boosting on large datasets.

#### 4. Missing Data Handling
Automatically learns a **default branch direction** for missing values by trying both sides and picking whichever reduces loss more.

#### 5. Pruning
Grows trees fully (depth-first), then **prunes back** branches that don't provide sufficient gain $\Delta L$, keeping the model lean and generalized.

In [ ]:
%pip install xgboost
# ==========================================
# CELL 14: XGBOOST (EXTREME GRADIENT BOOSTING)
# ==========================================
# Note: If you get a ModuleNotFoundError, run: !pip install xgboost
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV

print("Training XGBoost...")

# Tuning the number of trees to find the peak performance without overfitting
param_grid = {'n_estimators': [10, 50, 100, 150, 200, 300]}

# Define the Grid Search
# tree_method='hist' is the secret weapon that makes XGBoost run incredibly fast
grid_xgb = GridSearchCV(
    XGBClassifier(random_state=42, tree_method='hist', n_jobs=-1), 
    param_grid, 
    cv=3, 
    return_train_score=True, 
    n_jobs=-1
)

# Fit the model to the scaled training data
grid_xgb.fit(X_train, y_train)

# Output results using the universal function
plot_independent_dashboard("XGBoost", grid_xgb, 'n_estimators')

### K-Nearest Neighbors (Distance-Based Models)

*A distance-based "lazy learner" — it skips formula-building during `.fit()` and simply memorizes the entire training dataset instead.*

#### 1. Choose K
Select the number of neighbors $K$ to consider (e.g., `K=5`).

#### 2. Calculate Distance
For a new data point, compute the Euclidean distance to **every** training point:

$$d(x, x') = \sqrt{\sum_{i=1}^{n} (x_i - x'_i)^2}$$

#### 3. Find Nearest Neighbors
Sort all distances and select the **top K closest** points.

#### 4. Majority Vote (Prediction)
The majority class among the K neighbors is the final prediction:

$$\hat{y} = \arg\max_{k} \sum_{i=1}^{K} \mathbf{1}[y_i = k]$$

(e.g., if `K=5` and 4 neighbors are Class 1 → predict Class 1)

In [ ]:
param_grid = {'n_neighbors': [25,26, 27, 28,29,30,31,32,33,34,35]}

# Define grid_knn
grid_knn = GridSearchCV(
    KNeighborsClassifier(), 
    param_grid, 
    cv=3, 
    return_train_score=True, 
    n_jobs=-1,
)

# Fit the model
grid_knn.fit(X_train, y_train)

# Output results
plot_independent_dashboard("K-Nearest Neighbors", grid_knn, 'n_neighbors')


print("--- Average CV Scores ---")
results_knn = grid_knn.cv_results_

# Loop through the parameters and their corresponding average scores
for i in range(len(results_knn['params'])):
    c_val = results_knn['params'][i]['n_neighbors']
    avg_score = results_knn['mean_test_score'][i] * 100 # Convert to percentage
    
    print(f"C = {c_val:<6} | Average Accuracy: {avg_score:.2f}%")

### MLPClassifier (Neural Network Models)

*MLP (Multi-Layer Perceptron) is a deep learning model with interconnected neurons organized into layers.*

#### 1. Input Layer
Each feature maps to one input node — data enters the network here.

#### 2. Hidden Layers (Weights & Biases)
Each node computes a weighted sum of its inputs:

$$z = (x_1 \cdot w_1) + (x_2 \cdot w_2) + \dots + b$$

where $w$ = weights and $b$ = bias.

#### 3. Activation Function
$z$ is passed through an activation function (e.g., ReLU) to introduce non-linearity, allowing the network to learn complex patterns:

$$a = \text{ReLU}(z) = \max(0, z)$$

#### 4. Output Layer
The final layer outputs predicted class probabilities using Softmax (multi-class) or Sigmoid (binary).

#### 5. Backpropagation (Learning)
After each prediction, the model computes the error, travels **backwards** through the network, and adjusts weights using an optimizer (e.g., `adam`) to minimize loss:

$$w \leftarrow w - \eta \cdot \frac{\partial L}{\partial w}$$

where $\eta$ is the learning rate and $L$ is the loss.

In [ ]:
from sklearn.neural_network import MLPClassifier
param_grid = {'alpha': [0.0001, 0.001, 0.01, 0.1, 1.0, 10.0]}
grid_nn = GridSearchCV(MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=500, random_state=42), 
                    param_grid, cv=3, return_train_score=True, n_jobs=-1)
grid_nn.fit(X_train, y_train)
plot_independent_dashboard("Neural Network", grid_nn, 'alpha')

print("--- Average CV Scores ---")
results_nn = grid_nn.cv_results_

# Loop through the parameters and their corresponding average scores
for i in range(len(results_nn['params'])):
    c_val = results_nn['params'][i]['alpha']
    avg_score = results_nn['mean_test_score'][i] * 100 # Convert to percentage
    
    print(f"C = {c_val:<6} | Average Accuracy: {avg_score:.2f}%")

### Support Vector Machines (Geometric/Margin-Based Models)

*A geometric classifier that finds the optimal boundary between classes by maximizing the margin between them.*

#### 1. Hyperplane
Finds a decision boundary that separates Class 0 from Class 1. In 2D it's a line; in higher dimensions, a plane.

#### 2. Maximizing the Margin
Selects the boundary that creates the **widest possible gap** between classes to improve generalization:

$$\text{Maximize} \quad \frac{2}{\|w\|}$$

where $w$ is the weight vector defining the hyperplane.

#### 3. Support Vectors
Only the **closest data points** to the boundary (support vectors) determine its position — all other points are ignored.

#### 4. Kernel Trick
If data isn't linearly separable, a **kernel** (e.g., `rbf`, `poly`) projects data into a higher dimension where separation becomes possible:

$$K(x, x') = e^{-\gamma \|x - x'\|^2} \quad \text{(RBF Kernel)}$$

#### 5. C Parameter (Margin vs. Accuracy)
Controls the trade-off between margin width and misclassification:

| C Value | Margin | Tolerance for Mistakes |
|---------|--------|------------------------|
| Small | Wide | Higher |
| Large | Narrow | Lower |

In [ ]:
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV

# 1. Apply PCA to keep 80% of the variance
pca_svm = PCA(n_components=0.80, random_state=42) 
X_train_SVM_PCA = pca_svm.fit_transform(X_train)
X_test_SVM_PCA = pca_svm.transform(X_test)

print("--- SVM-Specific PCA Complete ---")
print(f"Original features: {X_train.shape[1]}")
print(f"Reduced features for SVM: {X_train_SVM_PCA.shape[1]}")
print("Training SVM on reduced features... Please wait.")



In [ ]:
# 2. Setup the Grid Search for SVM
param_grid = {'C': [0.01, 0.1, 1, 10, 50]}

grid_svm_pca = GridSearchCV(
    SVC(probability=True, random_state=42), 
    param_grid, 
    cv=3, 
    return_train_score=True, 
    n_jobs=-1
)

# 3. Fit on the PCA-reduced training data
grid_svm_pca.fit(X_train_SVM_PCA, y_train)

# 4. THE JUPYTER VARIABLE HACK: 
# Temporarily assign the reduced test set to 'X_test' so the dashboard function
# uses the correct shape, then safely swap it back to the original!
X_test_original = X_test 
X_test = X_test_SVM_PCA 

plot_independent_dashboard("Support Vector Machine (80% PCA)", grid_svm_pca, 'C')

# Restore the original 131-feature X_test for your other models
X_test = X_test_original

In [ ]:
# Create a dictionary of your trained models
trained_models = {
    "K-Nearest Neighbors": grid_knn,
    "Decision Tree": grid,
    "Logistic Regression": grid_lr,
    "Random Forest": grid_rf,
    "Gradient Boosting": grid_gb,
    "Neural Network": grid_nn,
    "Naive Bayes": grid_nb,
    "SVM (with PCA)": grid_svm_pca,
    "XGBoost": grid_xgb
}

print("🏆 MODEL LEADERBOARD (By Cross-Validation Accuracy) 🏆")
print("-" * 55)

# Extract the best score from each and sort them
leaderboard = []
for name, model in trained_models.items():
    # .best_score_ is the highest Test Accuracy found during GridSearch
    leaderboard.append((name, model.best_score_ * 100))

# Sort from highest to lowest
leaderboard.sort(key=lambda x: x[1], reverse=True)

# Print the results
for rank, (name, score) in enumerate(leaderboard, 1):
    print(f"{rank}. {name:<25} | Score: {score:.2f}%")

### Full Support Vector Machines (SVM) Work (More Accurate)

In [ ]:
# from sklearn.svm import SVC
# # Testing a narrow C range to save time
# param_grid = {'C': [0.01, 0.1, 1, 10, 50, 100]}
# grid_svm = GridSearchCV(SVC(probability=True, random_state=42), param_grid, cv=3, return_train_score=True, n_jobs=-1)
# grid_svm.fit(X_train, y_train)
# plot_independent_dashboard("SVM", grid_svm, 'C')